# Build an MCP server

- Core MCP Concepts
    - MCP servers can provide three main types of capabilities:
    - Resources: File-like data that can be read by clients (like API responses or file contents)
    - Tools: Functions that can be called by the LLM (with user approval)
    - Prompts: Pre-written templates that help users accomplish specific tasks

ref: [A Simple MCP Weather Server written in Python](https://github.com/modelcontextprotocol/quickstart-resources/tree/main/weather-server-python)

## Logging in MCP Servers
When implementing MCP servers, be careful about how you handle logging:

**For STDIO-based servers:** Never write to standard output (stdout). This includes:

- `print()` statements in Python
- `console.log()` in JavaScript
- `fmt.Println()` in Go
Similar stdout functions in other languages

Writing to stdout will **corrupt** the JSON-RPC messages and **break** your server.
(为了避免json解码失败，通信终端等异常行为)

For HTTP-based servers: Standard output logging `is fine` since it doesn’t interfere with HTTP responses.

In [2]:
# ❌ Bad (STDIO)
print("Processing request")

# ✅ Good (STDIO)
import logging
logging.info("Processing request")

Processing request


## System requirements

`Python 3.10` or higher installed You must use the Python `MCP SDK 1.2.0` or higher.

# set up enviroment

```conda
# 1. 创建项目目录
mkdir weather && cd weather

# 2. 创建 conda 环境
conda create -n mcp-weather python=3.11 -y
conda activate mcp-weather

# 3. 安装依赖（使用 pip）
pip install mcp[cli] httpx

```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("apikey.env")
print(os.getcwd())


'C:\\Users\\hhm18\\Desktop\\course'

In [11]:
from typing import Any
import httpx
from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("weather")

# Constants
NWS_API_BASE = "https://api.weather.gov"
USER_AGENT = "weather-app/1.0"

FastMCP类使用Python类型提示和文档字符串自动生成工具定义，从而便于创建和维护MCP工具。

---

美国国家气象局（NWS）是 美国政府 NOAA（National Oceanic and Atmospheric Administration） 下属的一个公共机构。
其提供的天气数据是 公共资源，依据美国联邦公开数据政策（Open Government Data Act），向公众开放使用。

In [9]:
import httpx

headers = {
    "User-Agent": "weather-mcp/1.0 (contact@h2mzzz163.com)",
    "Accept": "application/geo+json"
}

# 1. 获取经纬度对应的格点天气信息
response = httpx.get("https://api.weather.gov/points/39.7456,-97.0892", headers=headers)
data = response.json()
print(data)

# 2. 从格点信息中拿到具体预报 URL
forecast_url = data["properties"]["forecast"]
forecast = httpx.get(forecast_url, headers=headers).json()
print(forecast["properties"]["periods"][0]["detailedForecast"])


{'@context': ['https://geojson.org/geojson-ld/geojson-context.jsonld', {'@version': '1.1', 'wx': 'https://api.weather.gov/ontology#', 's': 'https://schema.org/', 'geo': 'http://www.opengis.net/ont/geosparql#', 'unit': 'http://codes.wmo.int/common/unit/', '@vocab': 'https://api.weather.gov/ontology#', 'geometry': {'@id': 's:GeoCoordinates', '@type': 'geo:wktLiteral'}, 'city': 's:addressLocality', 'state': 's:addressRegion', 'distance': {'@id': 's:Distance', '@type': 's:QuantitativeValue'}, 'bearing': {'@type': 's:QuantitativeValue'}, 'value': {'@id': 's:value'}, 'unitCode': {'@id': 's:unitCode', '@type': '@id'}, 'forecastOffice': {'@type': '@id'}, 'forecastGridData': {'@type': '@id'}, 'publicZone': {'@type': '@id'}, 'county': {'@type': '@id'}}], 'id': 'https://api.weather.gov/points/39.7456,-97.0892', 'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-97.0892, 39.7456]}, 'properties': {'@id': 'https://api.weather.gov/points/39.7456,-97.0892', '@type': 'wx:Point', 'cwa': '

虽然不要求 `API key`，但 强制要求添加自定义的 `User-Agent` 头。
这是因为：

NOAA 会拒绝掉没有 User-Agent 的请求。

---


## Hepler function 
helper functions for querying and formatting the data

In [ ]:
async def make_nws_request(url: str) -> dict[str, Any] | None:
    """Make a request to the NWS API with proper error handling."""
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/geo+json"
    }
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(url, headers=headers, timeout=30.0)
            response.raise_for_status()
            return response.json()
        except Exception:
            return None

def format_alert(feature: dict) -> str:
    """Format an alert feature into a readable string."""
    props = feature["properties"]
    return f"""
    Event: {props.get('event', 'Unknown')}
    Area: {props.get('areaDesc', 'Unknown')}
    Severity: {props.get('severity', 'Unknown')}
    Description: {props.get('description', 'No description available')}
    Instructions: {props.get('instruction', 'No specific instructions provided')}
    """

## Implementing tool execution


- 注册工具@mcp.tool：

    1. 把函数签名（参数类型、文档字符串、返回类型）封装成 MCP 的 Tool Schema

    2. 在服务启动时自动汇报给 MCP Host



In [12]:
@mcp.tool()
async def get_alerts(state: str) -> str:
    """Get weather alerts for a US state.

    Args:
        state: Two-letter US state code (e.g. CA, NY)
    """
    url = f"{NWS_API_BASE}/alerts/active/area/{state}"
    data = await make_nws_request(url)

    if not data or "features" not in data:
        return "Unable to fetch alerts or no alerts found."

    if not data["features"]:
        return "No active alerts for this state."

    alerts = [format_alert(feature) for feature in data["features"]]
    return "\n---\n".join(alerts)

@mcp.tool()
async def get_forecast(latitude: float, longitude: float) -> str:
    """Get weather forecast for a location.

    Args:
        latitude: Latitude of the location
        longitude: Longitude of the location
    """
    # First get the forecast grid endpoint
    points_url = f"{NWS_API_BASE}/points/{latitude},{longitude}"
    points_data = await make_nws_request(points_url)

    if not points_data:
        return "Unable to fetch forecast data for this location."

    # Get the forecast URL from the points response
    forecast_url = points_data["properties"]["forecast"]
    forecast_data = await make_nws_request(forecast_url)

    if not forecast_data:
        return "Unable to fetch detailed forecast."

    # Format the periods into a readable forecast
    periods = forecast_data["properties"]["periods"]
    forecasts = []
    for period in periods[:5]:  # Only show next 5 periods
        forecast = f"""
    {period['name']}:
    Temperature: {period['temperature']}°{period['temperatureUnit']}
    Wind: {period['windSpeed']} {period['windDirection']}
    Forecast: {period['detailedForecast']}
    """
        forecasts.append(forecast)

    return "\n---\n".join(forecasts)

1. @mcp.tool() 是一个 装饰器（decorator），用于告诉 MCP server：

    - “这是一个可以被外部模型（比如 ChatGPT、Claude、Gemini）调用的工具函数。”
    - 打包为标准的Tool Schema
        ```py
                {
        "name": "get_alerts",
        "description": "Get weather alerts for a US state.",
        "parameters": {
            "state": {"type": "string", "description": "Two-letter US state code"}
        }
        }

        ```
        - 供模型在运行时发现并调用。

In [14]:
def main():
    # Initialize and run the server
    mcp.run(transport='stdio')

# if __name__ == "__main__":
#     main()

最后运行脚本

```bsah
mcp run "weather.py"

python weather.py

```